# SQL Course - CASE Lesson
We can add a new calculated column and use CASE to switch between options.

The simple CASE form matches an exact value.
In this case we want to create a column that groups hospitals.
There is a final ELSE clause as a catch-all, for any hospital that does not meet any of the WHEN values.

In [5]:
SELECT
    ps.PatientId
    ,ps.Hospital
    ,CASE ps.Hospital
        WHEN 'PRUH' THEN 'Trust A'
        WHEN 'Oxleas' THEN 'Trust A'
        ELSE 'Trust B'
    END AS Trust
    ,ps.Ward
FROM
    dbo.PatientStay ps
ORDER BY
    ps.PatientId;

(44 rows affected)

PatientId | Hospital      | Trust   | Ward           
----------+---------------+---------+----------------
1         | Kingston      | Trust B | Dermatology    
2         | Kingston      | Trust B | Ophthalmology  
3         | PRUH          | Trust A | Day Surgery    
4         | Oxleas        | Trust A | Dermatology    
5         | Oxleas        | Trust A | Ophthalmology  
6         | Oxleas        | Trust A | Day Surgery    
7         | Kings College | Trust B | Dermatology    
8         | Kings College | Trust B | Ophthalmology  
9         | Kings College | Trust B | Day Surgery    
21        | Kingston      | Trust B | Day Surgery    
23        | Oxleas        | Trust A | Ophthalmology  
25        | Kings College | Trust B | Dermatology    
27        | Kings College | Trust B | Day Surgery    
29        | Kingston      | Trust B | Ophthalmology  
34        | Kings College | Trust B | Dermatology    
35        | Kings College | Trust B | Ophthalmology  
36      

The searched CASE form checks a condition.
In this case we want to group wards based on some complex logic.
This uses a 'match first' approach - the order of the WHEN THEN clauses may matter.

In [6]:
SELECT
    ps.PatientId
    ,ps.Hospital
    ,ps.Ward
    ,CASE
        WHEN ps.Ward LIKE '%Surgery' THEN 'Surgical'
        WHEN ps.Ward IN ('Accident', 'Emergency', 'Emergency Surgery') THEN 'A&E'
        WHEN ps.Tariff > 8 THEN 'ICU'
        ELSE 'General'
    END AS WardType
FROM
    dbo.PatientStay ps
ORDER BY
    WardType;

(44 rows affected)

PatientId | Hospital      | Ward            | WardType
----------+---------------+-----------------+---------
1         | Kingston      | Dermatology     | General 
2         | Kingston      | Ophthalmology   | General 
4         | Oxleas        | Dermatology     | General 
5         | Oxleas        | Ophthalmology   | General 
8         | Kings College | Ophthalmology   | General 
29        | Kingston      | Ophthalmology   | General 
34        | Kings College | Dermatology     | General 
35        | Kings College | Ophthalmology   | General 
37        | Kingston      | Dermatology     | General 
38        | PRUH          | Ophthalmology   | General 
25        | Kings College | Dermatology     | General 
110       | PRUH          | Ophthalmology   | General 
140       | Oxleas        | Ophthalmology   | General 
160       | Kings College | Dermatology     | General 
170       | Kings College | Ophthalmology   | General 
200       | Kingston      | Ophthalmology   |

Count rows where a condition is true using SUM with CASE.
Each row scores 1 if the condition is true, 0 if not. SUM then adds those scores up.
Note the calculation uses 100.0 (not 100) to ensure the division gives a decimal result rather than a whole number.

The row-by-row query uses CASE  to create a column with a value of 1 if the patient is in a surgical ward, 0 otherwise.

In [7]:
SELECT
    ps.Hospital
    ,ps.Ward
    ,CASE WHEN ps.Ward LIKE '%Surgery' THEN 1 ELSE 0 END AS IsPatientInSurgicalWard
FROM
    dbo.PatientStay ps
ORDER BY
    ps.PatientID;

(44 rows affected)

Hospital      | Ward            | IsPatientInSurgicalWard
--------------+-----------------+------------------------
Kingston      | Dermatology     | 0                      
Kingston      | Ophthalmology   | 0                      
PRUH          | Day Surgery     | 1                      
Oxleas        | Dermatology     | 0                      
Oxleas        | Ophthalmology   | 0                      
Oxleas        | Day Surgery     | 1                      
Kings College | Dermatology     | 0                      
Kings College | Ophthalmology   | 0                      
Kings College | Day Surgery     | 1                      
Kingston      | Day Surgery     | 1                      
Oxleas        | Ophthalmology   | 0                      
Kings College | Dermatology     | 0                      
Kings College | Day Surgery     | 1                      
Kingston      | Ophthalmology   | 0                      
Kings College | Dermatology     | 0                 

This query groups by hospital and sums those 0 or 1 values in the calculated column 
This has the effect of counting patients in each hospital that meet the CASE ... THEN 1  condition

In [8]:
SELECT
    ps.Hospital
    ,COUNT(*) AS NumberOfPatients
    ,SUM(CASE WHEN ps.Ward LIKE '%Surgery' THEN 1 ELSE 0 END) AS NumberOfPatientsInSurgery
    ,(100.0 * SUM(CASE WHEN ps.Ward LIKE '%Surgery' THEN 1 ELSE 0 END)) / COUNT(*) AS PercentageOfPatientsInSurgery
FROM
    dbo.PatientStay ps
GROUP BY
    ps.Hospital
ORDER BY
    ps.Hospital;

(4 rows affected)

Hospital      | NumberOfPatients | NumberOfPatientsInSurgery | PercentageOfPatientsInSurgery
--------------+------------------+---------------------------+------------------------------
Kings College | 13               | 5                         | 38.461538461538              
Kingston      | 12               | 4                         | 33.333333333333              
Oxleas        | 15               | 5                         | 33.333333333333              
PRUH          | 4                | 1                         | 25.000000000000              
(4 rows)

Total execution time: 00:00:00.020

## Optional advanced section

Work out which financial year a patient was admitted in. This assumes the financial year starts on 1st March.
For example, a patient admitted in January 2024 is in FY-2023-2024, and a patient admitted in March 2024 is in FY-2024-2025.

In [9]:
SELECT
    ps.PatientId
    ,ps.AdmittedDate
    ,CASE
        WHEN DATEPART(MONTH, ps.AdmittedDate) >= 3
            THEN CONCAT('FY-', DATEPART(YEAR, ps.AdmittedDate), '-', DATEPART(YEAR, ps.AdmittedDate) + 1)
        ELSE CONCAT('FY-', DATEPART(YEAR, ps.AdmittedDate) - 1, '-', DATEPART(YEAR, ps.AdmittedDate))
    END AS FinancialYear
FROM
    dbo.PatientStay ps
WHERE
    ps.Hospital = 'PRUH'
ORDER BY
    ps.AdmittedDate
    ,ps.PatientId;

(4 rows affected)

PatientId | AdmittedDate | FinancialYear
----------+--------------+--------------
3         | 2024-02-26   | FY-2023-2024 
110       | 2024-02-27   | FY-2023-2024 
38        | 2024-03-02   | FY-2024-2025 
433       | 2024-03-02   | FY-2024-2025 
(4 rows)

Total execution time: 00:00:00.020